# OSU Avatar Participant Aggregation and Merge — Female Image Scope

This notebook uses the 192-row automatic female-avatar feature file, keeps manual labels separate, creates participant-level first/last/count/proportion features, and left-merges them into the 190-row participant file.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, re
BASE=Path.cwd(); OUT=BASE/"outputs"; OUT.mkdir(exist_ok=True)
FEATURE=OUT/"avatar_features_female_all_192_automatic.csv"
MAPPING=BASE/"OSU_avatar_mapping_input.csv"
PARTICIPANT=BASE/"d_osu_final.csv"
for p in [FEATURE,MAPPING,PARTICIPANT]:
    if not p.exists(): raise FileNotFoundError(f"Missing required file: {p}")
feat=pd.read_csv(FEATURE); mapping=pd.read_csv(MAPPING); participant=pd.read_csv(PARTICIPANT)
print(feat.shape,mapping.shape,participant.shape)

In [ ]:
PAT=re.compile(r"^(?:avatar_)?(?P<type>[^_]+)_(?P<gender>[^_]+)_(?P<emotion>[^_]+)_(?P<color>[^_]+)_(?P<attire>[^_]+)_(?P<logo>[^_]+)_(?P<variant>v(?P<variant_number>\d+))$",re.I)
def parse_name(x):
    if pd.isna(x) or not str(x).strip(): return {}
    s=Path(str(x)).stem.removeprefix("avatar_"); m=PAT.match(s)
    if not m: return {}
    d={k:v.lower() for k,v in m.groupdict().items()}
    return {"avatar_name":s,"female":int(d["gender"]=="female"),"logo":int(d["logo"]!="nologo"),"happy":int(d["emotion"]=="happy"),"formal_binary":int(d["attire"]=="formal"),"gender":d["gender"],"emotion":d["emotion"],"color":d["color"],"attire":d["attire"],"logo_code":d["logo"],"variant":d["variant"],"variant_number":int(d["variant_number"])}

In [ ]:
lookup=feat.drop_duplicates("avatar_name").set_index("avatar_name").to_dict("index")
metrics=["mean_saturation_0_255","mean_brightness_0_255","realism_texture_proxy_0_100","craftsmanship_complexity_proxy_0_100","human_likeness_pixel_proxy_0_100","symmetry_proxy_0_1","entropy_bits","edge_density","local_texture"]
rows=[]
for _,r in mapping.iterrows():
    seq=[Path(str(r[c])).stem.removeprefix("avatar_") for c in ["avatar_1","avatar_2","avatar_3"] if pd.notna(r[c]) and str(r[c]).strip()]
    first=seq[0] if seq else None
    final=Path(str(r["final_avatar_name_source"])).stem.removeprefix("avatar_") if pd.notna(r["final_avatar_name_source"]) and str(r["final_avatar_name_source"]).strip() else (seq[-1] if seq else None)
    out={"Version.ID":r["Version.ID"],"User.ID":r["User.ID"],"first_avatar_name":first,"final_avatar_name":final,"last_avatar_name":final,"n_avatar_events":len(seq),"n_distinct_avatars":len(set(seq)) if seq else 0,"changed_avatar":int(len(set(seq))>1) if seq else np.nan}
    for prefix,nm in [("first",first),("final",final),("last",final)]:
        p=parse_name(nm)
        for k in ["female","logo","happy","formal_binary","gender","emotion","color","attire","logo_code","variant","variant_number"]: out[f"{prefix}_{k}"]=p.get(k,np.nan)
        met=lookup.get(nm,{}) if nm else {}
        for k in metrics: out[f"{prefix}_{k}"]=met.get(k,np.nan)
        out[f"{prefix}_female_image_metrics_available"]=int(nm in lookup) if nm else np.nan
    parsed=[parse_name(n) for n in seq]
    for k in ["female","logo","happy","formal_binary"]:
        vals=[p[k] for p in parsed if k in p]; out[f"count_{k}"]=sum(vals) if vals else np.nan; out[f"proportion_{k}"]=float(np.mean(vals)) if vals else np.nan
    for k in metrics:
        a=out.get(f"first_{k}"); b=out.get(f"final_{k}"); out[f"change_{k}"]=b-a if pd.notna(a) and pd.notna(b) else np.nan
    rows.append(out)
pf=pd.DataFrame(rows)
print(pf.shape)

In [ ]:
MANUAL_FULL = OUT / "manual_labels_all_168_separate.csv"

if not MANUAL_FULL.exists():
    raise FileNotFoundError(
        f"Missing required manual-label file: {MANUAL_FULL}. "
        "Place it in outputs/ before running this notebook."
    )

manual = pd.read_csv(MANUAL_FULL).sort_values("avatar_name").reset_index(drop=True)
assert len(manual) == 168
assert manual["avatar_name"].nunique() == 168

review = manual.iloc[np.linspace(0, len(manual) - 1, 28, dtype=int)].copy()
review.to_csv(OUT / "manual_labels_review_subset_28.csv", index=False)

print("Manual labels kept separate:", manual.shape)
print("Manual-label review subset saved:", review.shape)


In [ ]:
KEYS = ["Version.ID", "User.ID"]

assert len(feat) == 192 and feat.avatar_name.nunique() == 192
assert len(mapping) == 190 and not mapping.duplicated(KEYS).any()
assert len(participant) == 190 and not participant.duplicated(KEYS).any()
assert len(pf) == 190 and not pf.duplicated(KEYS).any()

merged = participant.merge(
    pf,
    on=KEYS,
    how="left",
    validate="one_to_one",
    indicator=True,
)

assert len(merged) == 190
assert (merged["_merge"] == "both").all()

merged = merged.drop(columns="_merge")

pf.to_csv(OUT / "participant_avatar_features_190.csv", index=False)
merged.to_csv(OUT / "d_osu_final_with_avatar_features_190.csv", index=False)

print("ALL PARTICIPANT MERGE CHECKS PASSED", merged.shape)
print("Manual-label subset:", OUT / "manual_labels_review_subset_28.csv")
print("Participant features:", OUT / "participant_avatar_features_190.csv")
print("Merged participant data:", OUT / "d_osu_final_with_avatar_features_190.csv")
